# Chẩn đoán trần F1 đạt được (Story #10 scoping — chưa tách Task)

Task #50 đo F1 ở ngưỡng mặc định 0.5 (F1 tốt nhất: XGBoost 0.312), rất xa mục tiêu 0.78. Notebook này **không huấn luyện lại** — chỉ dùng lại `models/*.pkl` (Task #49) và `orders_features_test.csv` để trả lời 2 câu hỏi trước khi quyết định phạm vi Story #10:

1. **F1 tối đa có thể đạt được qua tinh ngưỡng phân loại** (di chuyển dọc đường cong Precision-Recall hiện có) là bao nhiêu — có gần 0.78 không, hay bị chặn trần thấp hơn nhiều?
2. **Feature importance** của Random Forest/XGBoost — đặc trưng nào đang mang tín hiệu chính, có đặc trưng nào chiếm ưu thế bất thường (đặc biệt `order_purchase_month` — đã cảnh báo confound ở Story #6) không?

In [1]:
import joblib
import pandas as pd
from pathlib import Path
from sklearn.metrics import precision_recall_curve, average_precision_score, f1_score

df = pd.read_csv("../data/processed/orders_features_test.csv", low_memory=False)

df["is_delayed"] = df["is_delayed"].astype(bool)

bool_cols = ["payment_has_boleto", "payment_has_credit_card", "payment_has_debit_card",
             "payment_has_not_defined", "payment_has_voucher", "items_multi_seller"]
for col in bool_cols:
    df[col] = df[col].astype("boolean")

X_test = df.drop(columns=["order_id", "is_delayed"])
y_test = df["is_delayed"].astype(int)

models_dir = Path("../models")
models = {
    "logistic_regression": joblib.load(models_dir / "logistic_regression.pkl"),
    "random_forest": joblib.load(models_dir / "random_forest.pkl"),
    "xgboost": joblib.load(models_dir / "xgboost.pkl"),
}

print("X_test:", X_test.shape, " y_test:", y_test.shape)

X_test: (19289, 75)  y_test: (19289,)


## 1. F1 tối đa theo đường cong Precision-Recall (không huấn luyện lại)

Dùng `predict_proba` (xác suất lớp trễ) thay vì `.predict()` ở ngưỡng cố định 0.5. `precision_recall_curve` quét toàn bộ ngưỡng có thể — F1 tối đa trong số đó là **trần thật sự** đạt được bằng tinh ngưỡng, không cần huấn luyện lại model. `average_precision_score` (PR-AUC) tóm tắt cả đường cong, không phụ thuộc ngưỡng nào — dùng để so sánh chất lượng tín hiệu giữa các model.

In [2]:
threshold_rows = []
for name, model in models.items():
    y_score = model.predict_proba(X_test)[:, 1]
    precision, recall, thresholds = precision_recall_curve(y_test, y_score)

    # precision/recall co 1 phan tu nhieu hon thresholds (diem cuoi ung voi threshold=inf)
    f1_per_threshold = 2 * precision[:-1] * recall[:-1] / (precision[:-1] + recall[:-1] + 1e-12)
    best_idx = f1_per_threshold.argmax()

    threshold_rows.append({
        "model": name,
        "f1_at_default_0.5": f1_score(y_test, (y_score >= 0.5).astype(int)),
        "max_f1_via_threshold": f1_per_threshold[best_idx],
        "best_threshold": thresholds[best_idx],
        "precision_at_best": precision[best_idx],
        "recall_at_best": recall[best_idx],
        "average_precision_pr_auc": average_precision_score(y_test, y_score),
    })

threshold_df = pd.DataFrame(threshold_rows).sort_values("max_f1_via_threshold", ascending=False).reset_index(drop=True)
threshold_df

,model,f1_at_default_0.5,max_f1_via_threshold,best_threshold,precision_at_best,recall_at_best,average_precision_pr_auc
0,xgboost,0.311967,0.353143,0.646973,0.286797,0.459425,0.280744
1,random_forest,0.267521,0.339378,0.320000,0.285403,0.418530,0.261143
2,logistic_regression,0.200802,0.223906,0.576642,0.152000,0.424920,0.148707


## 2. Feature importance — Random Forest & XGBoost

Top 15 đặc trưng mỗi model, kiểm tra riêng thứ hạng `order_purchase_month` (cảnh báo confound đã ghi ở Story #6: dữ liệu chỉ trải ~2 năm không trọn, tín hiệu có thể trùng với giai đoạn bất thường phát hiện trước đó, không phải quy luật mùa vụ tổng quát).

In [3]:
importance_frames = []
for name in ["random_forest", "xgboost"]:
    importances = models[name].feature_importances_
    imp_df = pd.DataFrame({
        "model": name,
        "feature": X_test.columns,
        "importance": importances,
    }).sort_values("importance", ascending=False)
    imp_df["rank"] = range(1, len(imp_df) + 1)
    importance_frames.append(imp_df)

importance_df = pd.concat(importance_frames, ignore_index=True)

for name in ["random_forest", "xgboost"]:
    print(f"--- Top 15 — {name} ---")
    print(importance_df[importance_df["model"] == name].head(15)[["rank", "feature", "importance"]].to_string(index=False))
    month_row = importance_df[(importance_df["model"] == name) & (importance_df["feature"] == "order_purchase_month")]
    print(f"order_purchase_month rank: {month_row['rank'].values[0]} / {len(X_test.columns)}, importance: {month_row['importance'].values[0]:.4f}")
    print()

--- Top 15 — random_forest ---
 rank                   feature  importance
    1   estimated_delivery_days    0.149319
    2      order_purchase_month    0.107643
    3        approval_gap_hours    0.103366
    4       items_total_freight    0.093655
    5       payment_total_value    0.085876
    6      items_total_weight_g    0.085171
    7         items_total_price    0.082737
    8 payment_value_credit_card    0.067497
    9  payment_max_installments    0.037574
   10      payment_value_boleto    0.023441
   11         customer_state_SP    0.015117
   12         customer_state_RJ    0.011493
   13   primary_seller_state_SP    0.008828
   14           items_num_items    0.008630
   15         customer_state_MG    0.008475
order_purchase_month rank: 2 / 75, importance: 0.1076

--- Top 15 — xgboost ---
 rank                 feature  importance
    1       customer_state_SP    0.077871
    2       customer_state_MG    0.053498
    3       items_num_sellers    0.051734
    4    order_pu

## 3. Lưu kết quả chẩn đoán

Lưu vào `models/` (commit git, không phải `.pkl`) để tham chiếu khi bàn phạm vi Story #10.

In [4]:
threshold_df.to_csv(models_dir / "threshold_diagnostics.csv", index=False)
importance_df.to_csv(models_dir / "feature_importance.csv", index=False)
print("Da luu models/threshold_diagnostics.csv va models/feature_importance.csv")

Da luu models/threshold_diagnostics.csv va models/feature_importance.csv
